# Practical Exam: House sales

RealAgents is a real estate company that focuses on selling houses.

RealAgents sells a variety of types of house in one metropolitan area.

Some houses sell slowly and sometimes require lowering the price in order to find a buyer.

In order to stay competitive, RealAgents would like to optimize the listing prices of the houses it is trying to sell.

They want to do this by predicting the sale price of a house given its characteristics.

If they can predict the sale price in advance, they can decrease the time to sale.


## Data

The dataset contains records of previous houses sold in the area.

| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
| house_id    | Nominal. </br> Unique identifier for houses. </br>Missing values not possible. |
| city        | Nominal. </br>The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton'. </br>Replace missing values with "Unknown". |
| sale_price  | Discrete. </br>The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.</br>Remove missing entries. |
| sale_date   | Discrete. </br>The date of the last sale of the house. </br>Replace missing values with 2023-01-01. |
| months_listed  | Continuous. </br>The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. </br>Replace missing values with mean number of months listed, to one decimal place. |
| bedrooms    | Discrete. </br>The number of bedrooms in the house. Any positive values greater than or equal to zero. </br>Replace missing values with the mean number of bedrooms, rounded to the nearest integer. |
| house_type   | Ordinal. </br>One of "Terraced" (two shared walls), "Semi-detached" (one shared wall), or "Detached" (no shared walls). </br>Replace missing values with the most common house type. |
| area      | Continuous. </br>The area of the house in square meters, rounded to one decimal place. </br>Replace missing values with the mean, to one decimal place. |


# Task 1

The team at RealAgents knows that the city that a property is located in makes a difference to the sale price. 

Unfortuntately they believe that this isn't always recorded in the data. 

Calculate the number of missing values of the `city`. 

 - You should use the data in the file "house_sales.csv". 

 - Your output should be an object `missing_city`, that contains the number of missing values in this column. 

In [90]:
# Use this cell to write your code for Task 1
import pandas as pd

house_sales = pd.read_csv('house_sales.csv')
print(house_sales['city'].unique())
missing_city = (house_sales['city'] == '--').sum()
print(missing_city)

['Silvertown' 'Riverford' 'Teasdale' 'Poppleton' '--']
73


# Task 2 

Before you fit any models, you will need to make sure the data is clean. 

The table below shows what the data should look like. 

Create a cleaned version of the dataframe. 

 - You should start with the data in the file "house_sales.csv". 

 - Your output should be a dataframe named `clean_data`. 

 - All column names and values should match the table below.


| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
| house_id    | Nominal. </br> Unique identifier for houses. </br>Missing values not possible. |
| city        | Nominal. </br>The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton' </br>Replace missing values with "Unknown". |
| sale_price  | Discrete. </br>The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.</br>Remove missing entries. |
| sale_date   | Discrete. </br>The date of the last sale of the house. </br>Replace missing values with 2023-01-01. |
| months_listed  | Continuous. </br>The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. </br>Replace missing values with mean number of months listed, to one decimal place. |
| bedrooms    | Discrete. </br>The number of bedrooms in the house. Any positive values greater than or equal to zero. </br>Replace missing values with the mean number of bedrooms, rounded to the nearest integer. |
| house_type   | Ordinal. </br>One of "Terraced", "Semi-detached", or "Detached". </br>Replace missing values with the most common house type. |
| area      | Continuous. </br>The area of the house in square meters, rounded to one decimal place. </br>Replace missing values with the mean, to one decimal place. |

In [91]:
# Use this cell to write your code for Task 2
from sklearn.impute import SimpleImputer

    
imputer = SimpleImputer(strategy='mean')
house_sales['months_listed'] = imputer.fit_transform(house_sales[['months_listed']]).round(1)
house_sales['area'] =house_sales['area'].astype('string').str.replace('sq.m.','',regex=False) 
house_sales['area'] = house_sales['area'].astype('float64').round(1)
replace_dict = {
    'Det.' : 'Detached',
    'Semi' : 'Semi-detached',
    'Terr.': 'Terraced'
}
house_sales['house_type'] = house_sales['house_type'].str.strip().replace(replace_dict)
house_sales['house_type'] = pd.Categorical(
    house_sales['house_type'],
    categories=["Terraced", "Semi-detached", "Detached"],
    ordered=True
)
house_sales['city'] = house_sales['city'].astype('string').str.replace('--','Unknown',regex=False).astype('category')

clean_data = house_sales
clean_data['sale_date'] = pd.to_datetime(clean_data['sale_date']).dt.normalize()
print(clean_data.info())
print(clean_data['sale_date'])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   house_id       1500 non-null   int64         
 1   city           1500 non-null   category      
 2   sale_price     1500 non-null   int64         
 3   sale_date      1500 non-null   datetime64[ns]
 4   months_listed  1500 non-null   float64       
 5   bedrooms       1500 non-null   int64         
 6   house_type     1500 non-null   category      
 7   area           1500 non-null   float64       
dtypes: category(2), datetime64[ns](1), float64(2), int64(3)
memory usage: 73.7 KB
None
0      2021-09-12
1      2021-01-17
2      2021-11-10
3      2020-04-13
4      2020-09-24
          ...    
1495   2022-02-17
1496   2020-10-10
1497   2022-11-01
1498   2021-04-03
1499   2021-12-06
Name: sale_date, Length: 1500, dtype: datetime64[ns]


# Task 3 

The team at RealAgents have told you that they have always believed that the number of bedrooms is the biggest driver of house price. 

Producing a table showing the difference in the average sale price by number of bedrooms along with the variance to investigate this question for the team.

 - You should start with the data in the file 'house_sales.csv'.

 - Your output should be a data frame named `price_by_rooms`. 

 - It should include the three columns `bedrooms`, `avg_price`, `var_price`. 

 - Your answers should be rounded to 1 decimal place.   

In [92]:
# Use this cell to write your code for Task 3
price_by_rooms = house_sales.groupby('bedrooms')['sale_price'].agg(avg_price='mean',var_price='var').reset_index().round(1)


# Task 4

Fit a baseline model to predict the sale price of a house.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “validation.csv” to predict new values based on your model. You must return a dataframe named `base_result`, that includes `house_id` and `price`. The price column must be your predicted values.

In [93]:
# Use this cell to write your code for Task 4
train = pd.read_csv('train.csv')
validation = pd.read_csv('validation.csv')
print(train['city'].unique())
print(validation['city'].unique())
from statsmodels.formula.api import ols
from sklearn.metrics import mean_squared_error
import numpy as np

baseline_model = ols('sale_price ~ bedrooms',data=train).fit()
pred = baseline_model.predict(validation)
base_result = validation[['house_id']].copy()
base_result['price'] = pred
base_result


['Teasdale' 'Silvertown' 'Poppleton' 'Riverford']
['Teasdale' 'Silvertown' 'Riverford' 'Poppleton']


,house_id,price
0,1331375,151438.233892
1,1630115,227427.812045
2,1645745,379406.968350
3,1336775,151438.233892
4,1888274,227427.812045
...,...,...
295,1986255,379406.968350
296,1896276,379406.968350
297,1758223,227427.812045
298,1752010,151438.233892


# Task 5

Fit a comparison model to predict the sale price of a house.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “validation.csv” to predict new values based on your model. You must return a dataframe named `compare_result`, that includes `house_id` and `price`. The price column must be your predicted values.

In [94]:
# Use this cell to write your code for Task 5
from sklearn.ensemble import RandomForestRegressor

train['city'] = train['city'].astype('category')
validation['city'] = validation['city'].astype('category')

train['house_type'] = pd.Categorical(
    train['house_type'],
    categories=["Terraced", "Semi-detached", "Detached"],
    ordered=True
)
validation['house_type'] = pd.Categorical(
    validation['house_type'],
    categories=["Terraced", "Semi-detached", "Detached"],
    ordered=True
)
features = ['city', 'months_listed', 'bedrooms', 'house_type', 'area']
X_train = pd.get_dummies(train[features])
y_train = train['sale_price']
X_test = pd.get_dummies(validation[features])
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
model = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
compare_result = validation[['house_id']].copy()
compare_result['price'] = y_pred

compare_result

,house_id,price
0,1331375,82514.970000
1,1630115,303422.124317
2,1645745,405866.412102
3,1336775,106244.939128
4,1888274,267492.438683
...,...,...
295,1986255,363136.505412
296,1896276,391657.493838
297,1758223,264375.210916
298,1752010,178991.683421
